 **✅ Chapter 7 체크포인트**

**1. VS Code의 원격 개발 기능이 ML 엔지니어에게 왜 필수적인지, GPU 서버 활용 관점에서 설명할 수 있으신가요?**  
  > **답변:** VS Code 원격 개발 기능(Remote Development)은 원격 GPU 서버에 에이전트를 설치하여 소스 코드 분석, 린팅, 디버깅 등 무거운 작업은 서버의 자원을 사용하여 실행하고, UI 렌더링만 로컬 컴퓨터로 가져옵니다. 이를 통해 대규모 데이터와 연산이 필요한 GPU 서버 환경에서도 지연 없이 로컬 에디터 수준의 쾌적하고 편안한 작업 환경을 제공하므로 ML 엔지니어에게 필수적입니다.

**2. 주피터 노트북의 세 가지 문제점(실행 순서, 리팩토링, 버전 관리)을 각각 어떻게 해결하는지 "하이브리드 솔루션"의 관점에서 설명해 보세요.**  
  > **답변:** 하이브리드 솔루션은 데이터 탐색과 시각화는 노트북에서 수행하되, 모델 정의나 학습 루프 같은 **핵심 로직은 별도의 파이썬(.py) 모듈로 분리**하고 %autoreload로 연동하는 방식입니다.
      * **실행 순서 해결**: 복잡한 로직이 .py 스크립트로 분리되어 위에서부터 순서대로 실행되므로, 노트북 셀의 뒤죽박죽 실행으로 인한 꼬임 문제가 줄어듭니다.
      * **리팩토링 해결**: 핵심 로직이 .py 파일에 있으므로 IDE가 제공하는 '이름 바꾸기(Rename Symbol)' 등 강력한 리팩토링 기능을 온전히 활용할 수 있습니다.
      * **버전 관리 해결**: 실행 결과가 섞여 무거운 .ipynb 파일 대신, 순수 코드만 담긴 .py 파일을 Git으로 관리하게 되므로 diff를 읽기 쉽고 단위 테스트도 용이해집니다.
    
**3. "결정적 빌드"가 보장되지 않았을 때, 팀 협업에서 어떤 문제가 발생하는지 경험이나 예시를 떠올려 보실 수 있나요?**  
  > **답변:** 단순히 pip install 명령어로만 환경을 구성하면, 팀원들이 설치하는 시점에 따라 서로 미묘하게 다른 버전의 하위 라이브러리가 설치될 수 있습니다. 이로 인해 "어제 내 컴퓨터에서는 잘 돌아갔는데, 오늘 팀원의 환경에서는 에러가 나는" 상황이 발생하거나, 버전 비호환 문제로 인해 알 수 없는 이유로 학습 결과나 성능 수치가 달라지는 치명적인 문제가 발생합니다.
     
**4. requirements.in과 requirements.txt의 역할 차이를 설명할 수 있으신가요?**  
  > **답변:** `requirements.in`: 개발자가 직접 사용하는 최상위 핵심 라이브러리(예: `torch`, `transformers`)와 최소한의 버전 제약 조건만 명시하는 뼈대 역할을 합니다.  
   `requirements.txt`: 컴파일 도구(예: `pip-tools`, `uv`)를 통해 `requirements.in`을 바탕으로 상호 호환되는 **모든 하위 의존성의 정확한 버전을 찾아 고정(Locking)한 파일**입니다. 이 파일을 공유해야만 누구든 언제 어디서나 100% 동일한 환경을 재현할 수 있습니다.

**✅ Chapter 8 체크포인트**

**1. 전통적인 PyTorch 학습 루프 대비 Lightning으로 전환했을 때 사라지는 코드(직접 안 써도 되는 것) 5가지를 나열할 수 있으신가요?**
> **답변:** 데이터와 모델을 GPU로 보내는 `.to("cuda")` 같은 기기 할당 코드가 사라집니다. 또한 `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` 등 수동 최적화 단계와 에포크를 순회하는 반복문(학습/검증 루프)을 직접 쓸 필요가 없습니다. 메트릭을 수동으로 합산하고 평균 내는 코드와 분산 학습을 위한 복잡한 래핑(wrapping) 코드 역시 `Trainer`에 위임되어 생략됩니다.

**2. `LightningDataModule`의 `setup` 메서드와 `train_dataloader` 메서드의 역할 차이를 설명할 수 있으신가요? 왜 데이터 분할과 DataLoader 생성을 분리하는 것이 중요한가요?**
> **답변:** `setup` 메서드는 데이터를 다운로드하고 학습, 검증, 테스트 세트로 분할 및 전처리하는 초기 작업을 담당합니다. 반면 `train_dataloader`는 이렇게 준비된 데이터를 배치 단위로 묶어 모델에 공급할 실제 `DataLoader` 객체를 반환합니다. 이를 분리하면 여러 모델에서 동일한 데이터 준비 로직을 쉽게 재사용할 수 있으며, 다중 GPU 분산 학습 시 데이터 분할 처리를 프레임워크가 안전하게 자동화할 수 있습니다.

**3. 이 장에서 배운 `_shared_step` 패턴을 자신의 프로젝트에 적용한다면, 공통 로직과 각 단계(학습/검증/테스트)에서 달라지는 부분은 각각 무엇인가요?**
> **답변:** `_shared_step` 패턴에는 순전파(forward), 손실(loss) 계산, 정확도 등 평가지표 측정과 같이 모든 학습 단계에서 동일하게 수행되는 핵심 연산 로직이 들어갑니다. 반면 학습, 검증, 테스트의 각 단계별 메서드에서는 이 공통 함수의 반환값을 받아 로깅(`self.log`)하는 방식만 달라집니다. (예: 학습 단계에서는 스텝마다 로그를 남기고, 검증 단계에서는 에포크 단위로 합산하여 프로그레스 바에 표시)

**4. `overfit_batches=1` 테스트가 실패했을 때 의심해야 할 원인 3가지와, 각각의 디버깅 방법을 설명할 수 있으신가요?**
> **답변:** 첫째, 모델 내에서 연산 그래프가 끊겨 그래디언트가 업데이트되지 않는 경우이므로 `loss.backward()`가 정상 작동하는지 확인해야 합니다. 둘째, 데이터와 레이블이 엉뚱하게 매핑된 경우일 수 있어 데이터 로더의 셔플링이나 인덱싱 로직을 점검해야 합니다. 셋째, 연산 과정에서 오버플로우나 0으로 나누는 문제로 인한 NaN 발생이 원인일 수 있으니 데이터 정규화 상태를 확인해 보세요.

**5. 8.13의 전체 코드를 여러분의 실제 프로젝트에 맞게 수정한다면, `LitClassifier.__init__`과 `MyDataModule.setup`에서 각각 무엇을 바꿔야 하는지 구체적으로 떠올려 보세요.**
> **답변:** `LitClassifier.__init__`에서는 프로젝트가 해결할 태스크에 맞춰 입력 데이터 차원과 최종 출력 클래스 수를 변경하고, 실제 적용할 딥러닝 아키텍처(예: ResNet, BERT 등)로 대체해야 합니다. `MyDataModule.setup`에서는 임의로 생성한 랜덤 텐서 대신 프로젝트의 실제 데이터(예: 로컬 이미지, CSV 파일 등)를 읽어오도록 수정하고, 그에 맞는 전처리 파이프라인과 데이터 분할 코드를 작성해야 합니다.

**✅ Chapter 9 체크포인트**

**1. GPU 선택의 3대 지표(메모리 대역폭, 연산 속도, 인터커넥트) 중 분산 학습에서 가장 중요한 것은 무엇이며, 그 이유를 "고속도로와 톨게이트" 비유로 설명할 수 있으신가요?**
   > **답변:** 분산 학습에서는 여러 GPU 간에 그래디언트를 교환해야 하므로 통신 속도를 결정하는 **인터커넥트**가 가장 중요합니다. 연산 속도와 대역폭이 아무리 넓은 고속도로라 하더라도, 데이터를 주고받는 통로(인터커넥트)가 좁은 톨게이트라면 결국 병목이 발생하여 전체 학습 속도가 느려지기 때문입니다.

**2. "시간당 비용"이 아니라 "실험당 비용"으로 GPU를 평가해야 하는 이유를 표의 숫자 예시를 활용하여 설명할 수 있으신가요?**  
   > **답변:** 시간당 대여료가 비싼 고성능 GPU(예: A100, H100)를 사용하면 모델 학습에 걸리는 절대적인 시간이 획기적으로 단축됩니다. 단가가 저렴한 구형 GPU를 오래 사용하는 것보다, 비싼 GPU로 단시간에 실험을 끝내는 것이 전체 프로젝트 누적 비용(실험당 비용)과 엔지니어의 대기 시간을 모두 줄이는 더 경제적인 선택입니다.

**3. 스팟 인스턴스를 안전하게 사용하기 위해 반드시 갖춰야 할 전제 조건은 무엇인가요?**
   > **답변:** 스팟 인스턴스는 일반 가격보다 매우 저렴한 대신 클라우드 제공업체가 필요시 언제든지 예고 없이 자원을 회수할 수 있습니다. 따라서 인스턴스가 갑자기 종료되더라도 그동안의 학습 진행 상황을 잃지 않고 이어서 학습할 수 있도록 **체크포인팅 자동화(Model Checkpointing)** 기능이 반드시 구현되어 있어야 합니다.

**4. 여러분이 실습에서 진행한 실험 중, W&B가 있었다면 어떤 점이 편리했을지 구체적으로 떠올려 보실 수 있나요?**  
   > **답변:** 하이퍼파라미터 변경 이력이나 모델의 성능 지표를 주피터 노트북이나 메모장에 수동으로 기록할 필요 없이 대시보드에 자동으로 정리됩니다. 다양한 실험 결과를 한눈에 비교하고 시각화할 수 있어 최적의 설정값을 찾기 쉬워지며, GPU 메모리 사용률 등 시스템 지표도 함께 모니터링할 수 있어 병목을 파악하는 데 큰 도움이 됩니다.

**✅ Chapter 10 체크포인트**

**1. 스토리지 계층 구조의 지연 시간 차이가 `DataLoader`의 `num_workers` 설정에 어떤 영향을 미치는지 설명할 수 있으신가요?**
> **답변:** GPU의 연산 속도에 비해 디스크나 네트워크에서 데이터를 가져오는 스토리지 지연 시간은 매우 깁니다. 따라서 `DataLoader`의 `num_workers`를 늘려 여러 프로세스가 다음 배치 데이터를 미리 메모리로 가져오게(Prefetch) 해야 합니다. 이를 통해 GPU가 연산 중일 때 다음 데이터가 이미 준비되어 있어 병목 현상과 GPU의 대기 시간을 최소화할 수 있습니다.

**2. "이진 데이터는 S3에, URL만 DB에"라는 원칙의 이유를 설명할 수 있으신가요? 이 원칙을 어겼을 때 어떤 문제가 발생하나요?**
> **답변:** 이미지나 비디오 같은 용량이 큰 이진 데이터를 데이터베이스에 직접 저장하면 쿼리 성능이 급격히 저하됩니다. 또한 전체 DB의 크기가 비대해져 백업 및 복제 작업이 매우 무거워지고 관리 비용이 크게 증가하는 문제가 발생합니다. 따라서 무거운 이진 데이터는 확장성이 좋은 S3 같은 객체 저장소에 두고, DB에는 접근 가능한 가벼운 URL만 저장하여 데이터베이스 본연의 성능을 유지해야 합니다.

**3. OLTP와 OLAP의 차이를 "행 지향 vs 열 지향"의 관점에서, 각각 어떤 쿼리에 적합한지 예시와 함께 설명할 수 있으신가요?**
> **답변:** OLTP는 행(Row) 지향 데이터베이스로 특정 레코드를 빠르게 읽고 쓰는 데 최적화되어 있어, "사용자 A의 모든 주문 내역 조회"와 같은 트랜잭션 쿼리에 적합합니다. 반면 OLAP는 열(Column) 지향 데이터베이스로 대량의 데이터를 집계하고 분석하는 데 유리합니다. 필요한 열만 따로 읽어와 처리할 수 있어, "지난 30일간 등록된 전체 이미지의 평균 해상도 계산"과 같은 대규모 분석 쿼리에 훨씬 빠른 속도와 높은 압축률을 제공합니다.

**✅ Chapter 11 체크포인트**

**1. 레이블링을 최소화하는 4가지 전략(자기주도 학습, 증강, 합성 데이터, 사용자 피드백)을 각각 한 문장으로 요약하고, 자신의 프로젝트에 적용 가능한 것을 골라보실 수 있나요?**
> **답변:** 자기주도 학습은 데이터 자체 구조를 활용해 정답 없이 학습하며, 증강은 원본 특성을 유지한 채 변형을 가해 데이터를 늘립니다. 합성 데이터는 생성 모델이나 알고리즘으로 처음부터 정답이 있는 데이터를 만들고, 사용자 피드백은 실제 서비스 사용 기록을 레이블로 활용합니다. 챗봇이나 텍스트 프로젝트라면 비용이 적게 드는 **합성 데이터(LLM 활용)** 방식을 가장 우선적으로 적용해 볼 수 있습니다.

**2. 수동 레이블링의 5단계 프로세스 중 "골드 표준 데이터셋"은 왜 필요한가요? 이것 없이 외주를 맡기면 어떤 문제가 생기나요?**
> **답변:** 골드 표준 데이터셋은 여러 레이블러들이 작성한 결과물의 일치도와 품질을 객관적으로 평가하기 위한 정답 기준점 역할을 합니다. 이 기준과 가이드라인 없이 무작정 외주를 맡기면 작업자마다 주관적인 해석이 개입되어 데이터 일관성이 크게 훼손됩니다. 결과적으로 막대한 비용만 들이고 실제 학습에는 사용할 수 없는 저품질 데이터셋을 얻게 됩니다.

**3. 약지도 학습이 수동 레이블링 대비 갖는 가장 큰 장점을 "유지보수" 관점에서 설명해 보세요. 레이블 체계가 변경되었을 때 각각 어떤 일이 벌어지나요?**
> **답변:** 데이터의 스키마나 분류 체계가 변경되었을 때 수동 방식은 기존의 모든 데이터를 처음부터 다시 사람이 레이블링해야 하는 막대한 비용이 발생합니다. 반면 약지도 학습은 개별 데이터가 아닌 '레이블링 규칙(함수)'을 코드로 관리합니다. 따라서 체계가 바뀌더라도 해당 레이블링 함수(코드)만 수정한 뒤 다시 실행하면, 수백만 개의 전체 데이터 레이블을 즉시 갱신할 수 있어 유지보수에 압도적으로 유리합니다.

**4. LLM을 이용한 합성 데이터 생성이 효과적인 상황과 주의해야 할 점은 무엇인가요?**
> **답변:** 챗봇 응답이나 Q&A 세트처럼 대규모의 텍스트 Fine-tuning 데이터가 단기간에 필요할 때 레이블링 비용을 '0'에 가깝게 줄일 수 있어 매우 효과적입니다. 하지만 생성 모델 특성상 환각(Hallucination)이나 편향이 섞인 잘못된 데이터를 생성할 위험이 있습니다. 따라서 생성된 데이터를 맹신하지 말고 사람이 직접 일부 샘플링하여 품질을 검증하는 단계를 반드시 거쳐야 합니다.

**✅ Chapter 12 체크포인트**

**1. 데이터 버전 관리의 4단계 중 자신의 현재 프로젝트는 어느 레벨에 해당하나요? 한 단계 올리려면 구체적으로 무엇을 해야 하나요?**
> **답변:** 보통 많은 초기 프로젝트는 데이터 관리 기록이 없는 Level 0(무관리) 상태에 머물러 있습니다. 이를 한 단계 올리려면 학습할 때마다 데이터 전체를 날짜별 폴더에 복사해 두는 Level 1(스냅샷) 방식을 적용할 수 있습니다. 궁극적으로는 대용량 데이터는 원격 저장소에 두고 메타데이터만 Git LFS로 추적하는 Level 2(코드로서의 데이터)로 넘어가는 것이 실무적으로 가장 권장됩니다.

**2. "코드 + 데이터 = 모델"이라는 공식에서, 데이터 버전 관리 없이 모델을 롤백하려 할 때 어떤 문제가 발생하는지 구체적인 시나리오를 그려보실 수 있나요?**
> **답변:** 최근 배포된 모델 성능에 문제가 생겨 1주일 전 잘 작동하던 코드 버전으로 롤백했다고 가정해 보겠습니다. 데이터가 버전 관리되지 않았다면, 그 1주일 사이 추가되거나 삭제된 현재의 데이터로 과거 코드를 학습시켜야 하므로 과거 모델을 똑같이 재현할 수 없습니다. 결과적으로 성능 하락의 원인이 코드 변경 때문인지, 데이터 오염 때문인지 파악할 길이 막혀버립니다.

**3. Level 2("코드로서의 데이터")의 "장부" 비유를 자신의 언어로 다시 설명할 수 있으신가요?**
> **답변:** 수백 기가바이트의 이진 파일(이미지, 오디오 등)을 직접 Git으로 버전 관리하는 것은 물리적으로 불가능합니다. 대신 무거운 실제 데이터는 S3 같은 별도의 창고(클라우드 스토리지)에 쌓아두고, 이 파일들의 정확한 위치(URL)와 목록을 적은 가벼운 텍스트 문서인 '장부'만 Git으로 관리하는 것을 의미합니다. 이렇게 하면 Git에서 장부의 과거 커밋만 불러와도 당시 창고의 데이터 상태를 완벽하게 파악하고 복원할 수 있습니다.

**4. 워크플로우 자동화가 필요해지는 시점은 언제인가요? "단순한 파이썬 스크립트"로 충분한 경우와, Airflow 같은 전문 도구가 필요한 경우를 구분할 수 있으신가요?**
> **답변:** 워크플로우 자동화는 매일 혹은 매시간 단위로 여러 단계의 작업(데이터 추출 → 특징 생성 → 모델 재학습 등)이 앞 단계의 성공을 전제로 순차적으로 실행되어야 할 때 필요합니다. 만약 실패 시 단순히 다시 실행(Rerun)해도 무방한 가벼운 작업이라면 로컬 PC에서의 파이썬 스크립트나 크론탭(Cron)만으로도 충분합니다. 반면, 작업 간의 복잡한 의존성(DAG) 관리가 필요하고 실패 지점부터의 재시도, 실패 알람, 시각화된 대시보드 모니터링이 필수적인 엔터프라이즈 환경이라면 Airflow 같은 전문 도구를 도입해야 합니다.

**✅ Day 2 통합 체크포인트 — 오늘 배운 것을 하나로 꿰기**

**1. 환경 점검 : 여러분이 지금까지 실습에 사용한 환경을 떠올려 보세요. 재현 가능한 환경 구축(결정적 빌드)이 되어 있었나요? 팀원에게 동일한 환경을 전달하려면 어떤 파일들이 필요한가요?**
> **답변:** 기존에는 단순히 `pip install`에 의존하여 시점마다 버전이 달라지는 등 재현성이 부족했을 수 있습니다. 결정적 빌드를 통해 완벽히 동일한 환경을 팀원에게 전달하려면, 핵심 라이브러리만 명시한 `requirements.in` 파일과 이를 바탕으로 모든 하위 의존성 버전을 꽉 묶어둔(Locking) `requirements.txt` 파일이 함께 필요합니다.

**2. 코드 리팩토링 계획 : 지금까지 작성한 PyTorch 학습 코드 중 하나를 골라, Lightning으로 리팩토링한다면 어떤 부분이 LightningModule 로, 어떤 부분이 Trainer 설정으로 분리될지 구체적으로 구상해 보세요.**
> **답변:** `LightningModule` 내부에는 모델 아키텍처, 손실 함수와 함께 `training_step`(순전파 및 로깅), `validation_step`(검증 메트릭 계산), `configure_optimizers`(옵티마이저 설정) 같은 핵심 딥러닝 로직이 들어갑니다. 반면 `.to("cuda")`와 같은 기기 할당, 다중 GPU 분산 학습, 16비트 혼합 정밀도 등의 하드웨어 및 엔지니어링 제어 코드는 모두 `Trainer`의 파라미터 한 줄로 위임되어 코드가 분리됩니다.

**3. 인프라 설계 : Day 1에서 기획한 ML 프로젝트를 실행한다면, GPU는 무엇을 선택하고, 클라우드는 어디를 쓰며, 실험 관리는 어떤 도구를 사용할지 구체적으로 계획해 보세요. 예상 비용은 얼마인가요?**
> **답변:** 프로젝트 성격에 따라 다르겠지만, 빠른 연산과 넓은 대역폭이 필요하다면 A100이나 H100을 선택하고, 가성비를 위해 Lambda Labs 같은 GPU 전문 클라우드를 우선 고려할 수 있습니다. 실험 관리는 하이퍼파라미터와 시스템 지표를 한눈에 시각화해 주는 W&B(Weights & Biases)를 사용할 계획입니다. 예상 비용은 단순히 '시간당 대여료'가 아닌, 고성능 GPU를 짧게 사용하여 전체 누적 비용을 낮추는 '실험당 비용' 관점에서 산정해야 합니다.

**4. 데이터 전략 : 같은 프로젝트의 데이터를 어떻게 확보할 것인가요? 레이블링을 최소화하는 전략 중 적용 가능한 것은 무엇이며, 데이터 버전 관리는 어느 레벨부터 시작할 것인가요?**
> **답변:** 데이터 확보 초기에는 수동 작업을 줄이기 위해 최신 LLM을 활용한 합성 데이터(Synthetic Data) 생성이나 데이터 증강 기법을 가장 먼저 적용해 볼 수 있습니다. 이후 실제 서비스가 돌아가면 사용자의 클릭이나 수정 같은 피드백을 암묵적 레이블로 수집하는 구조를 짤 것입니다. 데이터 버전 관리는 무관리(Level 0)에서 벗어나, 최소한 S3에 실제 데이터를 두고 Git LFS로 메타데이터 장부를 관리하는 Level 2(코드로서의 데이터) 수준으로 시작하는 것이 좋습니다.

**5. 비용 시뮬레이션 : 위 프로젝트의 학습에 필요한 GPU 시간을 대략적으로 추산하고, 클라우드 비용을 "시간당"과 "실험당" 두 관점에서 비교해 보세요. 스팟 인스턴스를 쓸 수 있는 조건이 갖춰져 있나요?**
> **답변:** 싼 GPU로 며칠씩 길게 학습하는 것보다 비싸고 강력한 GPU로 단시간에 끝내는 것이 총비용(실험당 비용)과 엔지니어의 대기 시간을 모두 줄여줍니다. 클라우드의 잉여 자원을 빌리는 스팟 인스턴스를 활용하면 비용을 최대 90%까지 아낄 수 있습니다. 단, 이 저렴한 인스턴스를 안전하게 쓰려면 서버가 회수되더라도 학습을 이어나갈 수 있도록 PyTorch Lightning 등을 활용한 자동 체크포인팅 시스템이 반드시 구축되어 있어야 합니다.